# Révision de croyances — contraction de noyau, expansion et révision AGM

**Navigation** : [README](./README.md) | [5-formel <<](./Argumentation-05-Formal-Verification-Python.ipynb) | [7-orchestration >>](./Argumentation-07-Orchestration-Python.ipynb)

**Objectifs d'apprentissage**

- Distinguer le **booléen** du JTMS (le rung 6-JTMS : une croyance est IN ou OUT) de l'**ensemble de croyances** de la logique AGM (une croyance est un élément d'un ensemble, révocable)
- Expliquer la **contraction de noyau** : retirer une croyance peut emporter avec elle les formules qui la **supportent**, et seulement celles-là
- Reconstruire la **révision de Levi** comme une contraction suivie d'une expansion, et vérifier sur un cas où une croyance est *remplacée* par sa négation
- Connaître la **frontière syntaxique réelle** du parseur propositionnel de Tweety : quels connectifs il accepte, et ce qu'il fait d'un connectif écrit à la façon humaine
- Distinguer deux défaillances de surface d'un organe : l'erreur **Java** (que le port traduit en `RuntimeError`) et l'erreur **JPype** (que le port ne traduit pas)

**Prérequis** : le rung [5-formel](./Argumentation-05-Formal-Verification-Python.ipynb) (syntaxe propositionnelle de Tweety, démarrage de la JVM) et le rung 6-JTMS (le régime booléen non-monotone). JVM Tweety et JPype requis — aucune clé, aucun LLM.

**Durée** : 40 min.

Le rung 6-JTMS maintient une **étiquette** par croyance : `IN`, `OUT` ou `UNDECIDED`. C'est un régime *booléen* : believing, c'est porter l'étiquette `IN`. Mais la logique AGM (Gärdenfors, 1988) ne raisonne pas sur des étiquettes : elle raisonne sur un **ensemble de formules** et sur trois opérations qui le transforment.

| Opération | Question à laquelle elle répond | Forme |
|-----------|-----------------------------------|-------|
| **Expansion** | que believing `f` ajoute-t-il à l'ensemble ? | `B + f` |
| **Contraction** | que reste-t-il si l'on cesse de croire `f` ? | `B − f` |
| **Révision** | que reste-t-il si l'on remplace `f` par `g` ? | `B ⊕ g` |

La révision est *définie* comme la composition des deux autres : on contracte d'abord la négation de la nouvelle croyance, on expande ensuite. Ce carnet mesure cette composition sur un moteur réel.

Le moteur est un **organe déjà présent dans le dépôt** : `argumentation_lib/_belief_revision_handler.py`, copie verbatim du service EPITA (registre de provenance : `argumentation_lib/NOTICE-EPITA`). Ce carnet ne réimplémente rien : il appelle l'organe, il mesure ce qu'il rend, et il documente — sans les maquiller — les deux défaillances de surface qu'il expose.

In [1]:
# Cellule [1] - Demarrage de la JVM Tweety via la shim argumentation_lib (fail-loud)
#
# Meme point d'entree canonique que le rung 5-formel : la JVM n'est demarree
# qu'une fois par processus (memoisee), et un echec leve une exception au lieu
# de laisser croire a un mode degrade.
import sys
from pathlib import Path

# Rendre argumentation_lib importable quel que soit le cwd d'execution.
candidate = Path.cwd()
aa_dir = None
for _ in range(6):
    if (candidate / "argumentation_lib" / "__init__.py").exists():
        aa_dir = candidate
        break
    if len(candidate.parts) <= 1:
        break
    candidate = candidate.parent
if aa_dir is None:
    raise RuntimeError(
        "Dossier argumentation_lib/ introuvable depuis %r. Executez le carnet "
        "depuis MyIA.AI.Notebooks/SymbolicAI/Argument_Analysis/." % Path.cwd())
sys.path.insert(0, str(aa_dir))

from argumentation_lib import initialize_jvm, is_jvm_started, SYMBOLIC_AI_DIR

jvm_ok = initialize_jvm(verbose=True)
if not jvm_ok or not is_jvm_started():
    raise RuntimeError(
        "JVM/Tweety non demarree via la shim. Verifiez jdk-17-portable/ et "
        "libs/*.jar sous MyIA.AI.Notebooks/SymbolicAI/Tweety/. Mode degrade "
        "(simulation) INTERDIT dans cette serie.")

import jpype

print(f"JVM operationnelle : {jpype.isJVMStarted()}")
print(f"Repertoire de travail apres initialisation : {Path.cwd().name}")
print(f"Tweety charge depuis le dossier : {Path(SYMBOLIC_AI_DIR).name}")

--- Initialisation Tweety ---
JDK portable: zulu17.50.19-ca-jdk17.0.11-win_x64
Bibliotheques natives: native/


JVM demarree avec 42 JARs.
JVM operationnelle : True
Repertoire de travail apres initialisation : Tweety
Tweety charge depuis le dossier : SymbolicAI


### Interpretation : la JVM est-elle reellement demarree ?

Trois lignes distinguent un **moteur** d'un **theatre**. Une reponse « True » sans les deux autres ne prouverait rien : un `print("simule")` passerait le meme test.

| Sortie | Ce qu'elle etablit |
|--------|--------------------|
| `JVM operationnelle : True` | `jpype.isJVMStarted()` confirme une JVM vivante |
| `Repertoire de travail apres initialisation : Tweety` | la shim s'est deplacee dans `SymbolicAI/Tweety/`, ou resident les JARs |
| `Tweety charge depuis le dossier : SymbolicAI` | le dossier `SymbolicAI/` a ete localise ; les JARs resident dans `SymbolicAI/Tweety/libs/` |

> `initialize_jvm()` **change de repertoire courant** : les chemins relatifs de la suite du carnet (`libs/`, `jdk-17-portable/`) sont donc relatifs a `SymbolicAI/Tweety/`, pas a la racine de la serie. C'est une convention du port, pas un effet de bord du notebook.

## 2. L'organe : une copie verbatim, et ce qu'elle expose

`argumentation_lib.get_belief_revision_handler_symbol()` renvoie la **classe** `BeliefRevisionHandler` sans l'instancier : l'accesseur est paresseux, et l'instanciation, elle, exige une JVM deja demarree. La classe expose deux operations, toutes deux exposees par des **chaines de caracteres** :

| Methode | Signature | Sens |
|---------|-----------|------|
| `revise` | `revise(belief_set: list[str], new_belief: str, method: str = "dalal") -> dict` | revision : on remplace une croyance par une autre |
| `contract` | `contract(belief_set: list[str], formula_to_remove: str) -> dict` | contraction : on cesse de croire une formule |

Les deux renvoient un dictionnaire : le resultat (`revised` / `contracted`), l'entree (`original`), l'operande (`new_belief` / `removed`), et des `statistics` (tailles avant/apres).

Le nom `Dalal` merite un avertissement. Le port documente dans son en-tete que **Tweety 1.28 n'expose aucune classe d'operateur de revision de Dalal** (le paquet `revops` n'existe pas) : `dalal` et `levi` empruntent donc **le meme chemin Levi**. Le carnet le verifie plus bas, plutot que de s'en remettre a l'ecart annonce.

In [2]:
# Cellule [3] - Obtention de l'organe et lecture de son contrat
from argumentation_lib import get_belief_revision_handler_symbol

handler = get_belief_revision_handler_symbol()()

print("classe de l'organe :", type(handler).__name__)
print("methodes de revision declarees :", sorted(handler.REVISION_METHODS))
print("operations publiques :", [n for n in dir(handler) if not n.startswith("_")])
print("module d'origine :", type(handler).__module__)

classe de l'organe : BeliefRevisionHandler
methodes de revision declarees : ['dalal', 'levi']
operations publiques : ['DefaultExpansion', 'KernelContraction', 'LeviRevision', 'PlBeliefSet', 'PlFormula', 'PlNegation', 'PlParser', 'Proposition', 'REVISION_METHODS', 'contract', 'revise']
module d'origine : argumentation_lib._belief_revision_handler


### Interpretation : ce que l'organe promet, et ce qu'il ne promet pas

| Sortie | Lecture |
|--------|---------|
| `classe de l'organe : BeliefRevisionHandler` | l'accesseur paresseux a bien renvoye la classe du port, pas une reimplementation locale |
| `methodes de revision declarees : ['dalal', 'levi']` | l'ensemble `REVISION_METHODS` **declare** deux methodes — mais declares n'est pas applique : on le verifie plus bas |
| `operations publiques : [... 'KernelContraction', 'LeviRevision', ... 'contract', 'revise']` | la surface **operationnelle** est volontairement etroite : deux operations (`contract`, `revise`). Les autres entrees ne sont pas des operations : ce sont les **poignees de classes Java** que l'organe a construites au chargement (parseur, ensemble de croyances, operateurs) |
| `module d'origine : argumentation_lib._belief_revision_handler` | le code vient de la **bibliotheque du depot**, pas du notebook |

> La surface operationnelle etroite est une **bonne nouvelle pedagogique** : le carnet mesure deux operations, pas une bibliotheque. Elle est aussi une **limite** : un agent qui veut choisir *comment* la revision remplace une croyance n'a pas de prise ici — le choix est fige dans l'implementation de Levi. Quant aux poignees de classes Java exposees comme attributs, elles sont une **fuite de surface** herditee du port : elles ne sont pas documentees comme API, et le carnet ne s'en sert pas.

## 3. La frontiere syntaxique reelle du parseur propositionnel

Avant d'appeler l'organe, il faut savoir ce qu'il **accepte** comme formule. Le rung [5-formel](./Argumentation-05-Formal-Verification-Python.ipynb) documente la syntaxe de Tweety (`!`, `&&`, `||`, `=>`, `<=>`). La question que ce carnet tranche par la mesure est autre : que se passe-t-il quand on ecrit les connectifs **a la facon humaine** ?

La cellule ci-dessous fait passer une liste de formulations candidates au parseur `PlParser` de l'organe et note, pour chacune, si elle est acceptee ou refusee.

In [3]:
# Cellule [4] - Ce que le parseur PL accepte et ce qu'il refuse (mesure directe)
import jpype

parser = jpype.JClass("org.tweetyproject.logics.pl.parser.PlParser")()

CANDIDATS = [
    "p",            # atome
    "!p",           # negation (symbole ASCII)
    "\u00acp",       # negation (symbole logique)
    "p&&q",         # conjonction (symbole Tweety)
    "p||q",         # disjonction (symbole Tweety)
    "p=>q",         # implication
    "p<=>q",        # equivalence
    "(p&&q)",       # parentheses
    "!(p&&q)",      # negation d'une conjonction
    "p & q",        # conjonction ecrite a la facon humaine
    "p&q",          # idem, sans espaces
    "p | q",        # disjonction ecrite a la facon humaine
    "p -> q",       # implication ecrite a la facon humaine
    "p AND q",      # pseudo-connectif en majuscules
]

acceptees, refusees = [], []
for texte in CANDIDATS:
    try:
        formule = parser.parseFormula(texte)
        acceptees.append((texte, str(formule)))
    except jpype.JException:
        refusees.append(texte)

print("ACCEPTEES :")
for texte, rendu in acceptees:
    print(f"  {texte!r:14} -> {rendu!r}")
print("REFUSEES :")
for texte in refusees:
    print(f"  {texte!r:14} -> ParserException")

ACCEPTEES :
  'p'            -> 'p'
  '!p'           -> '!p'
  '¬p'           -> '¬p'
  'p&&q'         -> 'p&&q'
  'p||q'         -> 'p||q'
  'p=>q'         -> '(p=>q)'
  'p<=>q'        -> '(p<=>q)'
  '(p&&q)'       -> 'p&&q'
  '!(p&&q)'      -> '!(p&&q)'
  'p AND q'      -> 'pANDq'
REFUSEES :
  'p & q'        -> ParserException
  'p&q'          -> ParserException
  'p | q'        -> ParserException
  'p -> q'       -> ParserException


### Interpretation : deux pieges, dont un silencieux

La frontiere mesuree n'est pas « la syntaxe de Tweety » : elle est **la syntaxe que l'organe sait transmettre au parseur**, et elle comporte deux pieges de natures opposees.

**Piege 1 — le connectif ecrit a la facon humaine est refuse bruyamment.** `p & q`, `p&q`, `p | q`, `p -> q` leve une `ParserException`. C'est le comportement desirable : l'erreur est explicite, et l'appelant sait qu'il a mal ecrit.

**Piege 2 — le connectif ecrit en majuscules est accepte silencieusement, et change le sens.** `p AND q` est **accepte**, et rendu `pANDq` : le parseur a lu la chaine comme **un seul atome** nomme `pANDq`. Rien n'a echoue, rien n'a signale ; mais la formule portee n'est plus du tout celle qu'on croyait ecrire. Une revision qui ferait `revise(B, "p AND q")` introduirait dans l'ensemble de croyances une proposition nommee `pANDq` — un atome qui n'a aucun rapport avec `p` ni avec `q`.

> La regle qui en decoule : en logique propositionnelle, un **identifiant** est une suite de caracteres alphanumeriques. `AND` n'est pas un mot reserve, c'est un fragment d'identifiant. Seuls les **symboles** (`!`, `&&`, `||`, `=>`, `<=>`) sont des connectifs.

**Ce que la mesure dit du reste.** Les connectifs `&&` et `||` sont acceptes, l'implication `=>` aussi, et la negation s'ecrit `!` ou `¬`. Le refus de `&` et `|` n'est donc pas une restriction du moteur : c'est la difference entre le symbole du connectif (`&&`) et une approximation humaine de ce symbole (`&`).

## 4. La contraction de noyau : ce que « cesser de croire » emporte

La contraction de noyau (`kernel contraction`) ne retire pas l'atome demande : elle retire **le noyau minimal** de formules qui le derivait. Formulation operationnelle : on cherche le plus petit sous-ensemble `K` de l'ensemble de croyances tel que `K` entail la formule a retirer ; on retire `K`.

Deux consequences, et c'est tout l'interet pedagogique de l'operation :

1. si l'atome n'est entail par **aucune** formule de l'ensemble, le noyau est vide : **rien** n'est retire ;
2. si l'atome est entail par une formule **composee**, c'est cette formule-la qui part — l'atome etait un sous-element, pas un membre.

La cellule ci-dessous declenche la contraction sur trois formes qui se comportent differemment, plus le cas d'un atome absent.

In [4]:
# Cellule [5] - Contraction de noyau sur trois formes et un cas limite
print("--- contraction d'un atome SUPPORTE par une formule composee ---")
print("contract(['p&&q', 'r'], 'p') ->", handler.contract(["p&&q", "r"], "p")["contracted"])

print("--- contraction d'un atome NON entail par une disjonction ---")
print("contract(['p||q', 'r'], 'p') ->", handler.contract(["p||q", "r"], "p")["contracted"])

print("--- contraction d'un atome NON entail par une implication ---")
print("contract(['p=>q', 'r'], 'p') ->", handler.contract(["p=>q", "r"], "p")["contracted"])

print("--- contraction d'un atome qui n'appartient pas a l'ensemble ---")
print("contract(['p', 'q'], 'z') ->", handler.contract(["p", "q"], "z")["contracted"])

--- contraction d'un atome SUPPORTE par une formule composee ---
contract(['p&&q', 'r'], 'p') -> ['r']
--- contraction d'un atome NON entail par une disjonction ---
contract(['p||q', 'r'], 'p') -> ['p||q', 'r']
--- contraction d'un atome NON entail par une implication ---
contract(['p=>q', 'r'], 'p') -> ['(p=>q)', 'r']
--- contraction d'un atome qui n'appartient pas a l'ensemble ---
contract(['p', 'q'], 'z') -> ['p', 'q']


### Interpretation : trois formes, trois verdicts

Les trois verifications portent sur le meme atome (`p`) et le meme ensemble de taille comparable. Ce qui change, c'est la **forme de la formule qui contient `p`** — et cela change tout.

| Ensemble | Formule contenant `p` | `p` en est-elle un sous-element entail ? | Resultat | Ce qui s'est passe |
|----------|--------------------------|---------------------------------------------|----------|-------------------|
| `{p&&q, r}` | conjonction | **oui** — `p&&q` entail `p` | `['r']` | la formule entiere `p&&q` est partie : c'etait le noyau minimal |
| `{p||q, r}` | disjonction | **non** — `p||q` n'entail pas `p` | `['p||q', 'r']` | noyau vide : rien n'est retire |
| `{p=>q, r}` | implication | **non** — `p=>q` n'entail pas `p` | `['(p=>q)', 'r']` | noyau vide : rien n'est retire |
| `{p, q}` | aucune | l'atome est lui-meme membre | `['p', 'q']` | `z` n'est pas dans l'ensemble : le noyau est vide, rien n'est retire |

> **La distinction qui porte tout le sens de l'operation** : etre *mentionne* dans une formule ne suffit pas a etre *entail* par elle. `p` est mentionne dans `p||q` et dans `p=>q`, mais aucune de ces deux formules ne force `p`. Seul `p&&q` le fait. C'est pourquoi la premiere ligne perd une formule et les deux suivantes n'en perdent aucune.

Notez enfin la forme du resultat : le moteur **normalise** les formules qu'il rend (`p=>q` devient `(p=>q)`), et **trie** l'ensemble. La revision est donc une operation sur des ensembles normalises, pas sur des listes.

## 5. La revision de Levi : contracter la negation, puis expandre

La revision d'un ensemble `B` par une nouvelle croyance `g` se decompose :

1. **contraction** de `¬g` : on retire de `B` ce qui supportait l'absence de `g` — en pratique, ce qui derivait `¬g` ;
2. **expansion** de `g` : on ajoute `g` a l'ensemble residuel.

La consequence observable est la **substitution** : si `B` contenait `p` et que l'on revise par `!p`, le resultat contient `!p` et ne contient plus `p`. C'est le comportement signature d'une revision, et il se distingue d'une simple expansion (qui, elle, aurait garde `p`).

In [5]:
# Cellule [6] - Revision : ajout, substitution, formule implicative
print("--- revision par une croyance nouvelle (expansion seule) ---")
r = handler.revise(["p", "q"], "r")
print("revise(['p', 'q'], 'r') ->", r["revised"], "| stats:", r["statistics"])

print("--- revision qui REMPLACE une croyance par sa negation ---")
r = handler.revise(["p", "q"], "!p")
print("revise(['p', 'q'], '!p') ->", r["revised"], "| stats:", r["statistics"])
print("'p' est-il encore present ?", "p" in r["revised"])
print("'!p' est-il present ?", "!p" in r["revised"])

print("--- revision par une formule implicative ---")
r = handler.revise(["r"], "p=>q")
print("revise(['r'], 'p=>q') ->", r["revised"], "| stats:", r["statistics"])

print("--- revision par une croyance deja presente ---")
r = handler.revise(["p", "q"], "p")
print("revise(['p', 'q'], 'p') ->", r["revised"], "| stats:", r["statistics"])

--- revision par une croyance nouvelle (expansion seule) ---
revise(['p', 'q'], 'r') -> ['p', 'q', 'r'] | stats: {'original_size': 2, 'revised_size': 3}
--- revision qui REMPLACE une croyance par sa negation ---
revise(['p', 'q'], '!p') -> ['!p', 'q'] | stats: {'original_size': 2, 'revised_size': 2}
'p' est-il encore present ? False
'!p' est-il present ? True
--- revision par une formule implicative ---
revise(['r'], 'p=>q') -> ['(p=>q)', 'r'] | stats: {'original_size': 1, 'revised_size': 2}
--- revision par une croyance deja presente ---
revise(['p', 'q'], 'p') -> ['p', 'q'] | stats: {'original_size': 2, 'revised_size': 2}


### Interpretation : ce que la revision fait, et dans quel ordre

| Cas | Entree | Sortie | Lecture |
|-----|--------|--------|---------|
| Croyance nouvelle | `{p, q}` + `r` | `['p', 'q', 'r']` | rien a retirer, on ajoute : l'ensemble **grandit** (`original_size` 2 -> `revised_size` 3) |
| Substitution | `{p, 'q'}` + `!p` | `['!p', 'q']` | `p` **disparait**, `!p` apparait : c'est une revision, pas une expansion |
| Formule implicative | `{r}` + `p=>q` | `['(p=>q)', 'r']` | la formule est ajoutee, normalisee en `(p=>q)` par le moteur |
| Croyance deja presente | `{p, q}` + `p` | `['p', 'q']` | l'ensemble **ne change pas** (`original_size` 2 -> `revised_size` 2) : la revision est idempotente sur une croyance deja tenue |

Le deuxieme cas est le plus important : il est la **signature** de l'operation. Une expansion aurait produit `['p', 'q', '!p']` — un ensemble incoherent, contenant une croyance et sa negation. La revision, elle, passe par la contraction de `!p` : le noyau de `!p` dans `{p, q}` est `{p}` : `p` **contredit** `!p` (l'ensemble serait incoherent s'il portait les deux a la fois), et le mecanisme resout l'incoherence en **retirant** `p`. C'est ce qui distingue une revision d'une expansion, et c'est observable ici.

> Petit piege de lecture : la cellule affiche `'p' est-il encore present ? False` et `'!p' est-il present ? True`. C'est la preuve que l'operation a bien ete une **revision** et non une expansion.

## 6. Deux methodes nommees, un seul chemin de code

Le port declare deux methodes : `dalal` et `levi`. Son en-tete est explicite : Tweety 1.28 n'expose **aucune classe d'operateur de revision de Dalal**, et les deux noms empruntent donc le meme chemin Levi (contraction par noyau + expansion). Le carnet verifie l'egalite des resultats plutot que de s'y fier. La version des JARs n'est pas figee dans le depot (`libs/` est ignore par git) : la mesure vaut pour le jeu de JARs present a l'execution, pas pour une version figee.

Le meme en-tete annonce une autre chose : la methode passee n'est **pas validee**. Une methode inconnue n'est pas refusee — le chemin Levi est pris, et la valeur passee est recopiee telle quelle dans la cle `method` du resultat.

In [6]:
# Cellule [7] - dalal et levi convergent ; une methode inconnue n'est pas refusee
dalal = handler.revise(["p", "q"], "r", method="dalal")
levi = handler.revise(["p", "q"], "r", method="levi")

print("dalal ->", dalal["revised"])
print("levi  ->", levi["revised"])
print("resultats identiques :", dalal["revised"] == levi["revised"])

print("--- methode inconnue ---")
inconnu = handler.revise(["p"], "q", method="inconnu")
print("revise(['p'], 'q', method='inconnu') ->", inconnu["revised"])
print("cle 'method' rapportee :", inconnu["method"])
print("methode declaree par l'organe :", sorted(handler.REVISION_METHODS))

dalal -> ['p', 'q', 'r']
levi  -> ['p', 'q', 'r']
resultats identiques : True
--- methode inconnue ---
revise(['p'], 'q', method='inconnu') -> ['p', 'q']
cle 'method' rapportee : inconnu
methode declaree par l'organe : ['dalal', 'levi']


### Interpretation : deux declarations, un comportement

**Sur les deux methodes** : `dalal` et `levi` rendent le meme ensemble `['p', 'q', 'r']`. La distinction nominale ne correspond donc a aucune difference de comportement dans cette version de Tweety — c'est exactement ce que dit l'en-tete du port, et la mesure le confirme plutot que de le presumer.

**Sur la methode inconnue** : `method='inconnu'` **n'est pas refuse**. L'appel aboutit, l'ensemble revise est `['p', 'q']` (le chemin Levi a bien ete pris), et la cle `method` du resultat vaut la chaine `'inconnu'` — recopiee, pas validee. L'enumeration `REVISION_METHODS` est donc une **documentation**, pas un **garde-fou**.

> Consequence d'ingenierie : un appelant qui passe une methode mal orthographiee ne le saura pas. La seule protection disponible est de comparer soi-meme la methode a `handler.REVISION_METHODS` **avant** l'appel. Le carnet ne corrige pas le port — le fichier est une copie verbatim, et modifier une copie verbatim casse son contrat de provenance (`NOTICE-EPITA`). Le defaut est donc **documente**, pas corrige.

## 7. Deux defaillances de surface, a ne pas confondre

L'organe enveloppe ses appels Java dans un `try/except jpype.JException`, et traduit toute exception Java en `RuntimeError`. Cette traduction est **incomplete** : une seconde famille d'erreurs existe, qui n'est pas une exception Java du tout, et qui traverse donc l'enveloppe telle quelle.

La cellule ci-dessous fait passer a l'organe une famille de formules decomplexees, et note pour chacune le **type** d'exception obtenue. Deux families doivent apparaitre :

- `RuntimeError` — le **parseur** a refuse la chaine (`ParserException` Java) ;
- `TypeError` — la chaine a ete acceptee, mais la **dispatch JPype** est ambigue sur l'operateur appele.

In [7]:
# Cellule [8] - Matrice des formes de formules : quelle erreur, et de quelle famille
FORMULES = ["p", "!p", "p=>q", "p<=>q", "p&&q", "p||q", "!(p&&q)"]

print("=== revise(ensemble, nouvelle croyance) ===")
for f in FORMULES:
    try:
        resultat = handler.revise(["r"], f)
        print(f"  {f!r:12} -> OK       revised={resultat['revised']}")
    except RuntimeError as e:
        print(f"  {f!r:12} -> RuntimeError  {str(e)[:60]}")
    except TypeError as e:
        print(f"  {f!r:12} -> TypeError    {str(e)[:60]}")

print("=== contract(ensemble, formule a retirer) ===")
for f in FORMULES:
    try:
        resultat = handler.contract(["r", "s"], f)
        print(f"  {f!r:12} -> OK       contracted={resultat['contracted']}")
    except RuntimeError as e:
        print(f"  {f!r:12} -> RuntimeError  {str(e)[:60]}")
    except TypeError as e:
        print(f"  {f!r:12} -> TypeError    {str(e)[:60]}")

=== revise(ensemble, nouvelle croyance) ===
  'p'          -> OK       revised=['p', 'r']
  '!p'         -> OK       revised=['!p', 'r']
  'p=>q'       -> OK       revised=['(p=>q)', 'r']
  'p<=>q'      -> OK       revised=['(p<=>q)', 'r']
  'p&&q'       -> TypeError    Ambiguous overloads found for org.tweetyproject.beliefdynami
  'p||q'       -> TypeError    Ambiguous overloads found for org.tweetyproject.beliefdynami
  '!(p&&q)'    -> OK       revised=['!(p&&q)', 'r']
=== contract(ensemble, formule a retirer) ===
  'p'          -> OK       contracted=['r', 's']
  '!p'         -> OK       contracted=['r', 's']
  'p=>q'       -> OK       contracted=['r', 's']
  'p<=>q'      -> OK       contracted=['r', 's']
  'p&&q'       -> TypeError    Ambiguous overloads found for org.tweetyproject.beliefdynami
  'p||q'       -> TypeError    Ambiguous overloads found for org.tweetyproject.beliefdynami
  '!(p&&q)'    -> OK       contracted=['r', 's']


### Interpretation : deux familles d'echec, une seule. traduite

La matrice fait apparaitre une **frontiere nette** :

| Forme | `revise` | `contract` | Famille |
|-------|----------|------------|---------|
| `p` (atome) | OK | OK | — |
| `!p` (negation) | OK | OK | — |
| `p=>q` (implication) | OK | OK | — |
| `p<=>q` (equivalence) | OK | OK | — |
| `!(p&&q)` (negation d'une conjonction) | OK | OK | — |
| `p&&q` (conjonction) | **`TypeError`** | **`TypeError`** | dispatch JPype |
| `p||q` (disjonction) | **`TypeError`** | **`TypeError`** | dispatch JPype |

Deux remarques, dans cet ordre.

**1. Ce qui fonctionne est plus large que ce qui casse.** Les formes composees ne sont pas toutes refusees : une negation, une implication et une equivalence passent sans probleme, y compris `!(p&&q)`, dont le contenu est une conjonction. Ce n'est donc pas la **complexite** de la formule qui declenche l'echec, mais la **classe Java** de la formule.

**2. L'echec n'est pas la negation de l'organe.** Une `TypeError` n'est pas une `jpype.JException` : l'enveloppe `except jpype.JException` du port ne la capture pas, et elle remonte **brute** a l'appelant, sans le message `Belief revision failed: ...` que le port ajoute aux erreurs Java. Un appelant qui n'attrape que `RuntimeError` voit une exception **inattendue**.

> **Le mecanisme, tel que le message de JPype l'atteste — et ce qu'il n'atteste pas.** L'echec `TypeError: Ambiguous overloads` est bien produit par la resolution de surcharge de JPype : la liste des surcharges candidates est tronquee a 60 caracteres par le carnet, donc le detail des signatures n'apparait pas dans la sortie commitee. Ce que la sortie **etablit** : l'appel est ambigu du point de vue de JPype. Ce qu'elle **n'etablit pas** : l'heritage de classes (`Conjunction` et `Disjunction` heritant de `AssociativePlFormula`, qui implemente `java.util.Collection`) ni la liste exacte des deux surcharges `revise(PlBeliefSet, PlFormula)` et `revise(PlBeliefSet, PlFormula[])` — **ces trois faits restent une hypothese coherente avec le message, non mesuree dans ce carnet** (le jeu de JARs n'est pas versionne dans le depot, et ce siege n'a pas de JVM). Ce qui ne change pas pour l'etudiant : l'echec vient de la **passerelle JPype**, pas du port — et c'est le port qui herite de la limite.

## 8. Un operateur nomme « Random » : la stabilite mesuree

L'operateur de contraction utilise s'appelle `RandomKernelContractionOperator` : son nom annonce un comportement aleatoire. Le carnet ne s'y fie pas — il mesure. Dix contractions identiques et dix revisions identiques, sur le meme ensemble, doivent rendre **une seule** sortie distincte si le comportement est deterministe dans les conditions de l'execution.

In [8]:
# Cellule [9] - Stabilite mesuree sur dix appels identiques
contractions = {tuple(handler.contract(["p&&q", "r"], "p")["contracted"]) for _ in range(10)}
revisions = {tuple(handler.revise(["p", "q"], "r")["revised"]) for _ in range(10)}

print("contraction ['p&&q', 'r'] - p : sorties distinctes sur 10 appels =", len(contractions))
print("  valeur :", sorted(contractions))
print("revision ['p', 'q'] - r      : sorties distinctes sur 10 appels =", len(revisions))
print("  valeur :", sorted(revisions))

contraction ['p&&q', 'r'] - p : sorties distinctes sur 10 appels = 1
  valeur : [('r',)]
revision ['p', 'q'] - r      : sorties distinctes sur 10 appels = 1
  valeur : [('p', 'q', 'r')]


### Interpretation : le comportement observe, sans extrapolation

Sur les dix appels, **une seule sortie distincte** dans les deux cas. Dans les conditions de l'execution — une JVM unique, un meme jeu de JARs — le comportement est donc **deterministe**, malgre le nom de la classe.

Ce que la mesure **ne dit pas** : elle ne dit pas que le comportement sera identique sur une autre version de Tweety, sur un autre jeu de JARs, ou avec un ensemble de formules dont les noyaux multiples offriraient plusieurs choix. Le nom de la classe reste un avertissement de conception : l'algorithme est *concu* pour pouvoir choisir un noyau parmi plusieurs. La mesure etablit le comportement **observe ici**, pas une garantie.

> Un notebook qui conclut « l'operateur est deterministe » serait plus fort que la mesure. La formulation honnete est celle-ci : « dix appels identiques rendent une seule sortie distincte ; l'alerte du nom de classe reste documentee ».

## Exercice 1 — Expliquer une contraction

**Objectif** : ecrire une fonction qui, pour un ensemble de croyances et une formule a retirer, rend le resultat de la contraction **et** la raison pour laquelle le noyau est ce qu'il est.

**Consigne** : la fonction doit renvoyer un dictionnaire avec les cles `contracted` (la liste rendue par l'organe) et `kernel_reason` (une phrase expliquant pourquoi le noyau est vide ou non vide).

**Indice** : le noyau est non vide si et seulement si une formule de l'ensemble **entaille** la formule a retirer. Une conjonction entail ses composants ; une disjonction et une implication non.

**Etape 1** : appelez `handler.contract(...)` et recuperez la cle `contracted`.
**Etape 2** : determinez si le noyau etait vide en comparant la taille du resultat a celle de l'entree (`statistics`).
**Etape 3** : redigez la phrase en fonction de ces deux elements.

In [9]:
# Exercice 1 - a completer
def contraction_expliquee(handler, ensemble, formule):
    """Rend la contraction et la raison de son noyau.

    A completer : renvoyer un dictionnaire avec les cles
    'contracted' (liste rendue par l'organe) et 'kernel_reason' (phrase).
    """
    # TODO etudiant : implementer contraction_expliquee
    pass


# Test de non-regression : le resultat attendu est mesure dans la section 4.
print("Exercice 1 : contraction_expliquee(handler, ['p&&q', 'r'], 'p') ->",
      contraction_expliquee(handler, ["p&&q", "r"], "p"))

Exercice 1 : contraction_expliquee(handler, ['p&&q', 'r'], 'p') -> None


## Exercice 2 — Reproduire la revision de Levi

**Objectif** : ecrire la revision de Levi **decomposition en deux temps** — contraction de la negation de la nouvelle croyance, puis expansion — et verifier sur un cas ou la revision remplace une croyance.

**Consigne** : la fonction `revision_levi(handler, ensemble, nouvelle)` doit renvoyer un dictionnaire avec les cles `contracted` (l'ensemble apres contraction de `!nouvelle`), `expanded` (l'ensemble apres ajout de `nouvelle`) et `final` (le resultat). Elle doit utiliser les deux operations de l'organe, pas la methode `revise`.

**Indice** : pour retracter la negation d'une formule `f`, il faut ecrire la negation **dans la syntaxe de l'organe** : le parseur accepte `!f`, et la forme `¬f` (negation logique) est acceptee aussi.

**Etape 1** : contraction — `handler.contract(ensemble, "!" + formule)`.
**Etape 2** : expansion — ajouter la nouvelle croyance au resultat de l'etape 1.
**Etape 3** : comparer `final` avec `handler.revise(ensemble, formule)["revised"]` sur un cas ou la revision remplace une croyance.

In [10]:
# Exercice 2 - a completer
def revision_levi(handler, ensemble, nouvelle):
    """Revision de Levi en deux temps : contraction de !nouvelle, puis expansion.

    A completer : renvoyer un dictionnaire avec les cles
    'contracted', 'expanded' et 'final'.
    """
    # TODO etudiant : implementer revision_levi
    pass


# Test de non-regression : sur {'p','q'} revise par '!p', l'organe rend ['!p','q'].
print("Exercice 2 : revision_levi(handler, ['p', 'q'], '!p') ->",
      revision_levi(handler, ["p", "q"], "!p"))

Exercice 2 : revision_levi(handler, ['p', 'q'], '!p') -> None


## Exercice 3 — Classer ce que l'organe sait digerer

**Objectif** : ecrire un **classificateur de robustesse** : pour une liste de formules candidates, dire lesquelles l'organe accepte, et pour les autres, dire **de quelle famille** est l'echec.

**Consigne** : la fonction `classer_formules(handler, candidats)` renvoie un dictionnaire avec deux cles : `accepted` (la liste des formules acceptees) et `rejected` (un dictionnaire formule -> type d'exception, `RuntimeError` ou `TypeError`).

**Indice** : les deux familles d'echec ne se rattrapent pas de la meme facon. `RuntimeError` = le parseur a refuse la chaine (une faute d'ecriture, corrigeable en changeant la chaine). `TypeError` = la chaine est valide mais la dispatch JPype est ambigue (une **limite** de la passerelle, non corrigeable depuis l'appelant).

**Etape 1** : pour chaque formule, appeler `handler.revise(["r"], formule)` dans un `try/except`.
**Etape 2** : capturer `RuntimeError` et `TypeError` **separement** — un `except Exception` ne distinguerait pas les deux familles, et perdrait l'information qui fait la valeur du classement.
**Etape 3** : classer chaque formule dans la cle correspondante.

In [11]:
# Exercice 3 - a completer
def classer_formules(handler, candidats):
    """Classe les formules selon ce que l'organe en fait.

    A completer : renvoyer un dictionnaire avec les cles
    'accepted' (liste) et 'rejected' (formule -> type d'exception).
    """
    # TODO etudiant : implementer classer_formules
    pass


# Test de non-regression : mesuré en section 7, sur la famille complete.
print("Exercice 3 : classer_formules(handler, ['p', 'p&&q', 'p=>q']) ->",
      classer_formules(handler, ["p", "p&&q", "p=>q"]))

Exercice 3 : classer_formules(handler, ['p', 'p&&q', 'p=>q']) -> None


## Recapitulatif et limites

**Ce que le carnet etablit**, sur l'organe `argumentation_lib/_belief_revision_handler.py` (copie verbatim EPITA, `NOTICE-EPITA`) :

| Point | Verdict mesure |
|-------|----------------|
| Contraction sur un atome entail par une conjonction | la formule **entiere** part (noyau minimal) |
| Contraction sur un atome mentionne mais non entail (disjonction, implication) | **rien** ne part (noyau vide) |
| Revision par `!p` sur un ensemble contenant `p` | `p` est **remplace** par `!p` — l'operation de revision, pas une expansion |
| `dalal` et `levi` | **memes resultats** : l'en-tete du port documente l'absence de classe de revision de Dalal, et la mesure confirme la convergence sur le jeu de JARs present |
| Methode inconnue | **acceptee silencieusement**, chemin Levi pris, valeur recopiee dans la cle `method` |
| Formules `p&&q` et `p||q` | **`TypeError` (dispatch JPype ambigue)** — et cette erreur n'est pas traduite en `RuntimeError` par le port |
| Connectifs ecrits `&`, `|`, `->` | **refuses** (ParserException) |
| `p AND q` | **accepte silencieusement** comme l'atome `pANDq` |
| Stabilite | dix appels identiques rendent **une seule** sortie distincte |

**Ce que le carnet ne pretend pas** : le port n'est pas modifie (c'est une copie verbatim : la corriger casserait son contrat de provenance), la stabilite observee n'est pas une garantie, et la frontiere de la surface de chaines est une propriete de la version de Tweety embarquee, pas une loi de la logique AGM.

**Suite de la serie** : le rung [7-orchestration](./Argumentation-07-Orchestration-Python.ipynb) replace ces operations dans le protocole multi-agents ; le rung [8-capstone](./Argumentation-08-Capstone-Python.ipynb) les resitue dans le pipeline complet.